# Camada Silver - Transformação e Limpeza de Dados

Este notebook realiza a transformação dos dados da camada Bronze para a camada Silver, incluindo:
- Validações de qualidade
- Limpeza de dados
- Tratamento de valores nulos
- Padronização de tipos
- Derivação de novas features
- Remoção de duplicatas


## 1. Configuração Inicial

In [8]:
import os
import json
from datetime import datetime

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

print(f"Java configurado em: {os.environ.get('JAVA_HOME')}")

Java configurado em: /usr/lib/jvm/java-17-openjdk-amd64


In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, isnull, upper, lower, trim,
    to_date, to_timestamp, desc, year, month, dayofmonth,
    datediff, sum as spark_sum, avg as spark_avg, 
    min as spark_min, max as spark_max, concat_ws,
    coalesce, cast, abs, round
)
spark = SparkSession.builder \
    .appName("cybersecurity_silver") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Sessão Spark criada com sucesso!")
print(f"Versão do Spark: {spark.version}")

Sessão Spark criada com sucesso!
Versão do Spark: 4.1.1


## 2. Leitura de Dados Bronze

In [10]:
# Carregar dados da camada Bronze
df_incidents = spark.read.parquet(f"/home/carlos/cybersecurity-breach-data-project/data/bronze/incidents_master.parquet")

## 3. Análise e Validação de Qualidade - Camada Bronze

In [11]:
# Análise de duplicatas em incident_id
total_registros = df_incidents.count()
incident_ids_unicos = df_incidents.select('incident_id').dropDuplicates().count()
incident_ids_duplicados = total_registros - incident_ids_unicos

print("="*80)
print("ANÁLISE DE DUPLICATAS - incident_id")
print("="*80)
print(f"Total de registros: {total_registros}")
print(f"incident_id únicos: {incident_ids_unicos}")
print(f"Registros com incident_id duplicado: {incident_ids_duplicados}")
print(f"Taxa de duplicação: {(incident_ids_duplicados/total_registros*100):.2f}%")

if incident_ids_duplicados > 0:
    print("\nIncident IDs duplicados:")
    df_incidents.groupBy('incident_id').count().filter(col('count') > 1).show()

ANÁLISE DE DUPLICATAS - incident_id
Total de registros: 850
incident_id únicos: 850
Registros com incident_id duplicado: 0
Taxa de duplicação: 0.00%


In [12]:
# Análise de valores nulos
print("="*80)
print("ANÁLISE DE VALORES NULOS")
print("="*80)

null_counts = df_incidents.select(
    [count(when(isnull(c), c)).alias(c) for c in df_incidents.columns]
).collect()[0]

null_dict = {}
for col_name in df_incidents.columns:
    null_count = null_counts[col_name]
    null_pct = (null_count / total_registros * 100) if total_registros > 0 else 0
    null_dict[col_name] = {'count': null_count, 'percentage': null_pct}
    
# Mostrar apenas colunas com nulos
null_summary = sorted(
    [(k, v['count'], v['percentage']) for k, v in null_dict.items() if v['count'] > 0],
    key=lambda x: x[2],
    reverse=True
)

for col_name, count_null, pct_null in null_summary:
    print(f"{col_name:.<40} {count_null:>5} registros ({pct_null:>6.2f}%)")

ANÁLISE DE VALORES NULOS
review_flag.............................   780 registros ( 91.76%)
industry_secondary......................   697 registros ( 82.00%)
attack_vector_secondary.................   639 registros ( 75.18%)
notes...................................   636 registros ( 74.82%)
data_source_secondary...................   464 registros ( 54.59%)
stock_ticker............................   438 registros ( 51.53%)
downtime_hours..........................   430 registros ( 50.59%)
attributed_group........................   368 registros ( 43.29%)
attribution_confidence..................   368 registros ( 43.29%)
attack_chain............................   275 registros ( 32.35%)
data_compromised_records................   248 registros ( 29.18%)
data_type...............................   248 registros ( 29.18%)


In [13]:
# Schema do DataFrame
print("="*80)
print("SCHEMA - df_incidents")
print("="*80)
df_incidents.printSchema()

SCHEMA - df_incidents
root
 |-- incident_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- company_revenue_usd: double (nullable = true)
 |-- country_hq: string (nullable = true)
 |-- industry_primary: string (nullable = true)
 |-- industry_secondary: string (nullable = true)
 |-- employee_count: long (nullable = true)
 |-- is_public_company: boolean (nullable = true)
 |-- stock_ticker: string (nullable = true)
 |-- incident_date: string (nullable = true)
 |-- incident_date_estimated: boolean (nullable = true)
 |-- discovery_date: string (nullable = true)
 |-- disclosure_date: string (nullable = true)
 |-- attack_vector_primary: string (nullable = true)
 |-- attack_vector_secondary: string (nullable = true)
 |-- attack_chain: string (nullable = true)
 |-- attributed_group: string (nullable = true)
 |-- attribution_confidence: string (nullable = true)
 |-- data_compromised_records: double (nullable = true)
 |-- data_type: string (nullable = true)
 |-- systems

## 4. Transformações da Camada Silver - Incidents

In [14]:
# Passo 1: Remover duplicatas exatas
df_incidents.dropDuplicates()

print(f"Registros após remoção de duplicatas exatas: {df_incidents.count()}")

Registros após remoção de duplicatas exatas: 850


In [15]:
# Passo 3: Tratamento de valores nulos estratégicos
df_incidents = df_incidents.withColumn(
    'stock_ticker',
    when(col('stock_ticker') == '', None).otherwise(col('stock_ticker'))
).withColumn(
    'attack_vector_secondary',
    when(col('attack_vector_secondary') == '', None).otherwise(col('attack_vector_secondary'))
).withColumn(
    'attack_chain',
    when(col('attack_chain') == '', None).otherwise(col('attack_chain'))
).withColumn(
    'attributed_group',
    when(col('attributed_group') == '', None).otherwise(col('attributed_group'))
).withColumn(
    'attribution_confidence',
    when(col('attribution_confidence') == '', None).otherwise(col('attribution_confidence'))
).withColumn(
    'data_compromised_records',
    when(col('data_compromised_records') < 0, None)
    .otherwise(col('data_compromised_records'))
).withColumn(
    'downtime_hours',
    when(col('downtime_hours') < 0, None)
    .otherwise(col('downtime_hours'))
).withColumn(
    'notes',
    when(col('notes') == '', None).otherwise(col('notes'))
)

print("Valores nulos tratados com sucesso!")

Valores nulos tratados com sucesso!


In [16]:
# Passo 4: Criar features derivadas
df_incidents = df_incidents.withColumn(
    'days_to_discovery',
    datediff(col('discovery_date'), col('incident_date'))
).withColumn(
    'days_to_disclosure', 
    datediff(col('disclosure_date'), col('incident_date'))
).withColumn(
    'days_discovery_to_disclosure',
    datediff(col('disclosure_date'), col('discovery_date'))
).withColumn(
    'incident_year',
    year(col('incident_date'))
).withColumn(
    'incident_month',
    month(col('incident_date'))
).withColumn(
    'incident_day',
    dayofmonth(col('incident_date'))
).withColumn(
    'has_attribution',
    when(col('attributed_group').isNotNull(), True).otherwise(False)
).withColumn(
    'is_high_quality',
    when(col('quality_grade').isin('Gold', 'Silver'), True).otherwise(False)
).withColumn(
    'has_downtime',
    when(col('downtime_hours').isNotNull() & (col('downtime_hours') > 0), True).otherwise(False)
).withColumn(
    'has_data_loss',
    when(col('data_compromised_records').isNotNull() & (col('data_compromised_records') > 0), True).otherwise(False)
).withColumn(
    'is_flagged_review',
    when(col('review_flag').isNotNull(), True).otherwise(False)
).withColumn(
    'revenue_range',
    when(col('company_revenue_usd') < 1e6, 'Small')
    .when(col('company_revenue_usd') < 1e8, 'Medium')
    .when(col('company_revenue_usd') < 1e9, 'Large')
    .otherwise('Enterprise')
).withColumn(
    'employee_range',
    when(col('employee_count') < 100, 'Micro')
    .when(col('employee_count') < 1000, 'Small')
    .when(col('employee_count') < 10000, 'Medium')
    .otherwise('Large')
)

print("Features derivadas criadas com sucesso!")

Features derivadas criadas com sucesso!


In [17]:
# Passo 5: Reordenar e selecionar colunas finais
df_incidents = df_incidents.select(
    'incident_id',
    'incident_year',
    'incident_month',
    'incident_day',
    'incident_date',
    'incident_date_estimated',
    'discovery_date',
    'disclosure_date',
    'days_to_discovery',
    'days_to_disclosure',
    'days_discovery_to_disclosure',
    'company_name',
    'country_hq',
    'company_revenue_usd',
    'revenue_range',
    'employee_count',
    'employee_range',
    'is_public_company',
    'stock_ticker',
    'industry_primary',
    'industry_secondary',
    'attack_vector_primary',
    'attack_vector_secondary',
    'attack_chain',
    'attributed_group',
    'attribution_confidence',
    'has_attribution',
    'data_compromised_records',
    'has_data_loss',
    'data_type',
    'systems_affected',
    'downtime_hours',
    'has_downtime',
    'data_source_primary',
    'data_source_secondary',
    'data_source_type',
    'confidence_tier',
    'quality_score',
    'quality_grade',
    'is_high_quality',
    'review_flag',
    'is_flagged_review',
    'notes',
    'created_at',
    'updated_at',
    'ingestion_timestamp'
)

print("Colunas reordenadas com sucesso!")
print(f"Total de colunas na Silver: {len(df_incidents.columns)}")

Colunas reordenadas com sucesso!
Total de colunas na Silver: 46


## 5. Validação de Qualidade - Camada Silver

In [18]:
# Verificar qualidade dos dados transformados
print("="*80)
print("VALIDAÇÃO DE QUALIDADE - SILVER")
print("="*80)

print(f"\nTotal de registros: {df_incidents.count()}")
print(f"Total de colunas: {len(df_incidents.columns)}")

print("\nSchema da tabela Silver:")
df_incidents.printSchema()

VALIDAÇÃO DE QUALIDADE - SILVER

Total de registros: 850
Total de colunas: 46

Schema da tabela Silver:
root
 |-- incident_id: string (nullable = true)
 |-- incident_year: integer (nullable = true)
 |-- incident_month: integer (nullable = true)
 |-- incident_day: integer (nullable = true)
 |-- incident_date: string (nullable = true)
 |-- incident_date_estimated: boolean (nullable = true)
 |-- discovery_date: string (nullable = true)
 |-- disclosure_date: string (nullable = true)
 |-- days_to_discovery: integer (nullable = true)
 |-- days_to_disclosure: integer (nullable = true)
 |-- days_discovery_to_disclosure: integer (nullable = true)
 |-- company_name: string (nullable = true)
 |-- country_hq: string (nullable = true)
 |-- company_revenue_usd: double (nullable = true)
 |-- revenue_range: string (nullable = false)
 |-- employee_count: long (nullable = true)
 |-- employee_range: string (nullable = false)
 |-- is_public_company: boolean (nullable = true)
 |-- stock_ticker: string (nul

In [19]:
# Verificar valores nulos na Silver
print("="*80)
print("VALORES NULOS - SILVER")
print("="*80)

null_counts_silver = df_incidents.select(
    [count(when(isnull(c), c)).alias(c) for c in df_incidents.columns]
).collect()[0]

null_dict_silver = {}
for col_name in df_incidents.columns:
    null_count = null_counts_silver[col_name]
    null_pct = (null_count / df_incidents.count() * 100) if df_incidents.count() > 0 else 0
    null_dict_silver[col_name] = {'count': null_count, 'percentage': null_pct}

# Mostrar colunas com nulos
null_summary_silver = sorted(
    [(k, v['count'], v['percentage']) for k, v in null_dict_silver.items() if v['count'] > 0],
    key=lambda x: x[2],
    reverse=True
)

if null_summary_silver:
    for col_name, count_null, pct_null in null_summary_silver:
        print(f"{col_name:.<45} {count_null:>5} ({pct_null:>6.2f}%)")
else:
    print("Nenhuma coluna com valores nulos!")

VALORES NULOS - SILVER
review_flag..................................   780 ( 91.76%)
industry_secondary...........................   697 ( 82.00%)
attack_vector_secondary......................   639 ( 75.18%)
notes........................................   636 ( 74.82%)
data_source_secondary........................   464 ( 54.59%)
stock_ticker.................................   438 ( 51.53%)
downtime_hours...............................   430 ( 50.59%)
attributed_group.............................   368 ( 43.29%)
attribution_confidence.......................   368 ( 43.29%)
attack_chain.................................   275 ( 32.35%)
data_compromised_records.....................   248 ( 29.18%)
data_type....................................   248 ( 29.18%)


In [20]:
# Verificar features derivadas
print("="*80)
print("VALIDAÇÃO DE FEATURES DERIVADAS")
print("="*80)

print(f"\nIncidents com dias_to_discovery calculado: {df_incidents.filter(col('days_to_discovery').isNotNull()).count()}")
print(f"Incidents com dias_to_disclosure calculado: {df_incidents.filter(col('days_to_disclosure').isNotNull()).count()}")
print(f"Incidents com atribuição: {df_incidents.filter(col('has_attribution') == True).count()}")
print(f"Incidents com perda de dados: {df_incidents.filter(col('has_data_loss') == True).count()}")
print(f"Incidents com downtime: {df_incidents.filter(col('has_downtime') == True).count()}")
print(f"Incidents flagged para revisão: {df_incidents.filter(col('is_flagged_review') == True).count()}")
print(f"Incidents com alta qualidade: {df_incidents.filter(col('is_high_quality') == True).count()}")

VALIDAÇÃO DE FEATURES DERIVADAS

Incidents com dias_to_discovery calculado: 850
Incidents com dias_to_disclosure calculado: 850
Incidents com atribuição: 482
Incidents com perda de dados: 602
Incidents com downtime: 420
Incidents flagged para revisão: 70
Incidents com alta qualidade: 693


In [21]:
# Distribuição de dados
print("\n" + "="*80)
print("DISTRIBUIÇÃO DE DADOS")
print("="*80)

print("\nIncidents por ano:")
df_incidents.groupBy('incident_year').count().orderBy('incident_year').show()

print("\nIncidents por país (Top 10):")
df_incidents.groupBy('country_hq').count().orderBy(desc('count')).limit(10).show()

print("\nIncidents por indústria primária:")
df_incidents.groupBy('industry_primary').count().orderBy(desc('count')).show()


DISTRIBUIÇÃO DE DADOS

Incidents por ano:
+-------------+-----+
|incident_year|count|
+-------------+-----+
|         2021|  131|
|         2022|  143|
|         2023|  166|
|         2024|  181|
|         2025|  229|
+-------------+-----+


Incidents por país (Top 10):
+----------+-----+
|country_hq|count|
+----------+-----+
|        US|  298|
|        GB|   90|
|        DE|   50|
|        CA|   48|
|        FR|   40|
|        AU|   37|
|        JP|   35|
|        IN|   25|
|        IT|   20|
|        BR|   17|
+----------+-----+


Incidents por indústria primária:
+----------------+-----+
|industry_primary|count|
+----------------+-----+
|              62|  142|
|              52|  128|
|              51|  127|
|           44-45|   81|
|           31-33|   79|
|              92|   52|
|              22|   38|
|              61|   37|
|              54|   36|
|           48-49|   23|
|              21|   21|
|              56|   16|
|              72|   16|
|              71|   14|
|

## 6. Salvar Dados Silver

In [ ]:
import os
import shutil

print("Salvando dados Silver...")

df_incidents.coalesce(1).write \
    .mode("overwrite") \
    .parquet("/home/carlos/cybersecurity-breach-data-project/data/silver/incidents_master")


for file in os.listdir("/home/carlos/cybersecurity-breach-data-project/data/silver/incidents_master"):
    if file.endswith(".parquet"):
        shutil.move(
            f"/home/carlos/cybersecurity-breach-data-project/data/silver/incidents_master/{file}",
            "/home/carlos/cybersecurity-breach-data-project/data/silver/incidents_master.parquet"
        )
        break


shutil.rmtree("/home/carlos/cybersecurity-breach-data-project/data/silver/incidents_master")

print("✓ Dados salvos em: /home/carlos/cybersecurity-breach-data-project/data/silver/incidents_master.parquet")
print(f"✓ Registros salvos: {df_incidents.count()}")

Salvando dados Silver...
✓ Dados salvos em: /home/carlos/cybersecurity-breach-data-project/data/silver/incidents_master.parquet
✓ Registros salvos: 850
